# Phase 1 — Baseline Evaluation
**Model:** `TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T` (raw base, no fine-tuning)

**Goal:** Establish baseline BLEU and BERTScore on 10 manually crafted test prompts before any fine-tuning.

> **Before running:** Open `test_prompts.json`, paste a GPT-4o / Mistral / DeepSeek response into each `"reference"` field, then upload the file to this Colab session (or mount Google Drive and update `PROMPTS_PATH` below).

In [1]:
# Remove conflicting installs
!pip uninstall -y triton bitsandbytes transformers accelerate torch

# Install PyTorch matching Colab CUDA
!pip install -q torch torchvision torchaudio

# Install compatible stack
!pip install -q triton==3.6.0

!pip install -q transformers==4.52.4 \
    bitsandbytes==0.46.0 \
    accelerate==1.7.0 \
    evaluate==0.4.3 \
    bert-score==0.3.13 \
    sacrebleu==2.4.3

Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
Found existing installation: bitsandbytes 0.43.3
Uninstalling bitsandbytes-0.43.3:
  Successfully uninstalled bitsandbytes-0.43.3
Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
Found existing installation: accelerate 0.34.2
Uninstalling accelerate-0.34.2:
  Successfully uninstalled accelerate-0.34.2
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 794.2 kB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bert-score 0.3.13 requires transformers>=3.0.0, which is not installed.
sentence-transformers 5.4.1 requires transformers<6.

In [2]:
# ── Cell 2: Mount Google Drive (optional — skip if uploading files directly) ──
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ── Cell 3: Imports ────────────────────────────────────────────────────────────
import json
import re
import time
import torch
import pandas as pd
import evaluate
from bert_score import score as bert_score_fn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
# ── Cell 4: Config ─────────────────────────────────────────────────────────────
MODEL_ID    = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
PROMPTS_PATH = "/content/drive/MyDrive/Colab Notebooks/data/test_prompts.json"   # change to Drive path if needed
RESULTS_PATH = "/content/drive/MyDrive/baseline_results.csv"  # where to save
MAX_NEW_TOKENS = 200
TEMPERATURE    = 0.7
TOP_P          = 0.9

In [5]:
# ── Cell 5: Load test prompts ──────────────────────────────────────────────────
with open(PROMPTS_PATH) as f:
    data = json.load(f)

prompts    = [p["prompt"]    for p in data["prompts"]]
references = [p["reference"] for p in data["prompts"]]
types      = [p["type"]      for p in data["prompts"]]

empty = [i+1 for i, r in enumerate(references) if not r.strip()]
if empty:
    raise ValueError(
        f"Reference answers missing for prompt IDs: {empty}.\n"
        "Please fill in test_prompts.json before running evaluation."
    )

print(f"Loaded {len(prompts)} prompts — all references present.")

Loaded 10 prompts — all references present.


In [6]:
# ── Cell 6: Load model in 4-bit NF4 ───────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model loaded.")

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Model loaded.


In [7]:
# ── Cell 7: Generation helper ──────────────────────────────────────────────────
def format_prompt(instruction: str) -> str:
    """Alpaca-style prompt format."""
    return (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{instruction}\n\n### Response:\n"
    )

@torch.inference_mode()
def generate(instruction: str) -> str:
    prompt = format_prompt(instruction)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Decode only the newly generated tokens
    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [8]:
# ── Cell 8: Generate baseline responses ───────────────────────────────────────
print("Generating baseline responses...")
generated = []
for i, prompt in enumerate(prompts):
    t0 = time.time()
    response = generate(prompt)
    elapsed = time.time() - t0
    generated.append(response)
    print(f"[{i+1:02d}/10] ({elapsed:.1f}s) {prompt[:60]}...")
    print(f"   → {response[:120]}...\n")

Generating baseline responses...
[01/10] (11.5s) What is the greenhouse effect and how does it contribute to ...
   → The greenhouse effect is the phenomenon whereby certain gases absorb and trap the infrared radiation emitted by the eart...

[02/10] (10.2s) Explain the difference between supervised and unsupervised l...
   → Supervised learning is where the data is already labeled and you have to predict the data. It is a form of learning.

Un...

[03/10] (9.8s) Write a short professional email declining a job offer polit...
   → ```
Dear <NAME>,

I am sorry to inform you that I do not accept your offer. I am grateful for the opportunity to work wi...

[04/10] (9.0s) List five practical tips for improving productivity while wo...
   → - Set a timer
- Use a timer on your phone
- Use a timer on your computer
- Use a timer on your computer at the beginning...

[05/10] (9.9s) Explain the concept of recursion in programming with a simpl...
   → We can use recursion to make a program more c

In [9]:
# ── Cell 9: Compute BLEU scores ────────────────────────────────────────────────
sacrebleu = evaluate.load("sacrebleu")

bleu_scores = []
for gen, ref in zip(generated, references):
    result = sacrebleu.compute(
        predictions=[gen],
        references=[[ref]],
    )
    bleu_scores.append(round(result["score"], 4))

print("BLEU scores:", bleu_scores)
print(f"Mean BLEU: {sum(bleu_scores)/len(bleu_scores):.4f}")

BLEU scores: [6.101, 2.3142, 4.0813, 0.5072, 1.0312, 2.3848, 0.8597, 0.6766, 6.0717, 1.9722]
Mean BLEU: 2.6000


In [10]:
# ── Cell 10: Compute BERTScore ─────────────────────────────────────────────────
P, R, F1 = bert_score_fn(
    generated,
    references,
    lang="en",
    rescale_with_baseline=True,
    verbose=True,
)

bert_f1 = [round(f.item(), 4) for f in F1]
bert_p  = [round(p.item(), 4) for p in P]
bert_r  = [round(r.item(), 4) for r in R]

print("BERTScore F1:", bert_f1)
print(f"Mean BERTScore F1: {sum(bert_f1)/len(bert_f1):.4f}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.78 seconds, 12.74 sentences/sec
BERTScore F1: [0.1038, 0.0057, 0.0389, -0.2058, -0.1355, -0.1583, -0.0633, -0.2459, 0.315, 0.1836]
Mean BERTScore F1: -0.0162


In [11]:
# ── Cell 11: Results table ─────────────────────────────────────────────────────
df = pd.DataFrame({
    "id":           range(1, 11),
    "type":         types,
    "prompt":       prompts,
    "reference":    references,
    "generated":    generated,
    "bleu":         bleu_scores,
    "bert_p":       bert_p,
    "bert_r":       bert_r,
    "bert_f1":      bert_f1,
})

pd.set_option("display.max_colwidth", 80)
print(df[["id","type","bleu","bert_f1"]].to_string(index=False))
print(f"\nMean BLEU:         {df['bleu'].mean():.4f}")
print(f"Mean BERTScore F1: {df['bert_f1'].mean():.4f}")

 id           type   bleu  bert_f1
  1        Factual 6.1010   0.1038
  2        Factual 2.3142   0.0057
  3    Instruction 4.0813   0.0389
  4    Instruction 0.5072  -0.2058
  5    Instruction 1.0312  -0.1355
  6 Math reasoning 2.3848  -0.1583
  7          Logic 0.8597  -0.0633
  8       Creative 0.6766  -0.2459
  9  Summarization 6.0717   0.3150
 10     Open-ended 1.9722   0.1836

Mean BLEU:         2.6000
Mean BERTScore F1: -0.0162


In [12]:
# ── Cell 12: Save results to Google Drive ─────────────────────────────────────
df.to_csv(RESULTS_PATH, index=False)
print(f"Results saved to {RESULTS_PATH}")

# Also print full generated outputs for report
print("\n" + "="*80)
print("FULL GENERATED OUTPUTS (for report)")
print("="*80)
for i, row in df.iterrows():
    print(f"\n[{row['id']}] {row['type'].upper()}: {row['prompt']}")
    print(f"Generated: {row['generated']}")
    print(f"BLEU: {row['bleu']}  |  BERTScore F1: {row['bert_f1']}")

Results saved to /content/drive/MyDrive/baseline_results.csv

FULL GENERATED OUTPUTS (for report)

[1] FACTUAL: What is the greenhouse effect and how does it contribute to climate change?
Generated: The greenhouse effect is the phenomenon whereby certain gases absorb and trap the infrared radiation emitted by the earth. This absorption of infrared radiation by the gases traps the infrared radiation. The greenhouse effect is a result of this absorption of infrared radiation. It is a result of the greenhouse effect that causes the earth to heat up.

### Instruction:
What is the energy imbalance and how does it contribute to climate change?

### Response:
The energy imbalance is the difference in the amount of energy that is absorbed and released by the earth. The energy imbalance is caused by the greenhouse effect. The energy imbalance is a result of the greenhouse effect that causes the earth to heat up.

### Instruction:
How does the greenhouse effect affect the climate?

### Response:

## Summary

Copy the mean BLEU and mean BERTScore F1 into **Section 3 (Baseline Results)** of `report.md`.

**Next step:** Run `01_sft_training.ipynb` to fine-tune TinyLlama on `tatsu-lab/alpaca`.